In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

MODEL_PATH = '/content/drive/MyDrive/Proiect_Licenta_XAUUSD/models/lstm_advanced_v1.h5'
CSV_PATH = '/content/drive/MyDrive/Proiect_Licenta_XAUUSD/data/processed/XAUUSD_with_indicators.csv'


In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import load_model
import plotly.graph_objects as plotly_go
from plotly.subplots import make_subplots

# Setup Paths
PROJECT_PATH = '/content/drive/MyDrive/Proiect_Licenta_XAUUSD'
MODEL_LSTM_PATH = f'{PROJECT_PATH}/models/lstm_advanced_v1.h5'
MODEL_RF_PATH = f'{PROJECT_PATH}/models/random_forest_v1.pkl' 
MODEL_XGB_PATH = f'{PROJECT_PATH}/models/xgboost_optimized_v1.pkl'
CSV_PATH = f'{PROJECT_PATH}/data/processed/XAUUSD_with_indicators.csv'

st.set_page_config(layout="wide", page_title="XAUUSD AI Trading")

@st.cache_resource
def load_all_models():
    lstm = load_model(MODEL_LSTM_PATH, compile=False)
    
    try:
        rf = joblib.load(MODEL_RF_PATH)
    except FileNotFoundError:
        st.error(f"Model RF not found: {MODEL_RF_PATH}")
        rf = None
        
    try:
        xgb = joblib.load(MODEL_XGB_PATH)
    except FileNotFoundError:
        st.error(f"Model XGBoost not found: {MODEL_XGB_PATH}")
        xgb = None
        
    return lstm, rf, xgb

@st.cache_data
def load_and_prepare_data():
    df = pd.read_csv(CSV_PATH, index_col=0, parse_dates=True)
    prediction_horizon = 3
    
    df['target_raw'] = np.log(df['close'].shift(-prediction_horizon) / df['close'])
    df = df.dropna()

    features = [
        'dist_ema_200', 'atr', 'bb_pct', 'fix_vix', 'is_vix_green', 
        'smi_oversold', 'smi_overbought', 'ret_1h', 'ret_3h', 'hour', 'day_of_week'
    ]
    smi_col = [c for c in df.columns if 'smi' in c and 'over' not in c and '_s' not in c][0]
    features.append(smi_col)

    scaler = MinMaxScaler(feature_range=(0, 1))
    X_scaled = scaler.fit_transform(df[features])

    time_steps = 48
    Xs_3d, Xs_2d, y_raw = [], [], []
    dates, closes = [], []

    for i in range(len(X_scaled) - time_steps):
        seq = X_scaled[i:(i + time_steps)]
        Xs_3d.append(seq)
        Xs_2d.append(seq[-1]) 
        
        y_raw.append(df['target_raw'].iloc[i + time_steps])
        dates.append(df.index[i + time_steps])
        closes.append(df['close'].iloc[i + time_steps])

    split = int(len(Xs_3d) * 0.8)
    
    return (np.array(Xs_3d[split:]), np.array(Xs_2d[split:]), 
            np.array(y_raw[split:]), dates[split:], closes[split:])


lstm_model, rf_model, xgb_model = load_all_models()
X_test_3d, X_test_2d, y_test_raw, test_dates, test_closes = load_and_prepare_data()

st.title("XAUUSD Algorithm Comparison Simulator")


st.sidebar.header("Strategy Parameters")
selected_model = st.sidebar.selectbox(
    "Select Trading Model", 
    ["LSTM (Deep Learning)", "Random Forest", "XGBoost"]
)

st.sidebar.markdown("---")

# Generate Predictions
if 'predictions' not in st.session_state:
    st.session_state.predictions = {}

if selected_model not in st.session_state.predictions:
    with st.spinner(f'Running inference for {selected_model}...'):
        if selected_model == "LSTM (Deep Learning)":
            preds = lstm_model.predict(X_test_3d, verbose=0).flatten()
        elif selected_model == "Random Forest":
            preds = rf_model.predict(X_test_2d)
        elif selected_model == "XGBoost":
            preds = xgb_model.predict(X_test_2d)
        st.session_state.predictions[selected_model] = preds

model_signals = st.session_state.predictions[selected_model]


st.sidebar.write("**Model Prediction Range:**")
min_pred = np.min(model_signals)
max_pred = np.max(model_signals)
st.sidebar.write(f"Min: {min_pred:.5f}")
st.sidebar.write(f"Max: {max_pred:.5f}")
st.sidebar.markdown("---")


if selected_model == "LSTM (Deep Learning)":
    st.sidebar.write("**LSTM Parameter (Long-Only):**")
    confidence_threshold = st.sidebar.slider("Confidence Threshold (Prob)", 0.50, 0.99, 0.54, 0.01)
else:
    st.sidebar.write("**RF / XGBoost Parameters:**")
    long_threshold = st.sidebar.number_input(
        "BUY Threshold (Positive Return)", 
        min_value=-0.10000, 
        max_value=0.10000, 
        value=0.00050, 
        step=0.00010, 
        format="%.5f"
    )
    short_threshold = st.sidebar.number_input(
        "SELL Threshold (Negative Return)", 
        min_value=-0.10000, 
        max_value=0.10000, 
        value=-0.00050, 
        step=0.00010, 
        format="%.5f"
    )

# Backtest Engine
in_trade_until = 0
trades_taken = 0
winning_trades = 0
equity = 1.0
equity_curve = []
buy_signals_idx = []
sell_signals_idx = []

take_profit = 0.0030
stop_loss = -0.0010

for i in range(len(model_signals)):
    if i < in_trade_until:
        equity_curve.append(equity)
        continue

    # Evaluate Entry
    trade_direction = 0  
    
    if selected_model == "LSTM (Deep Learning)":
        if model_signals[i] > confidence_threshold:
            trade_direction = 1
    else: 
        if model_signals[i] > long_threshold:
            trade_direction = 1
        elif model_signals[i] < short_threshold:
            trade_direction = -1

    # Evaluate Exit
    if trade_direction != 0:
        if trade_direction == 1:
            buy_signals_idx.append(i)
        elif trade_direction == -1:
            sell_signals_idx.append(i)
            
        trades_taken += 1
        
        trade_ret = y_test_raw[i] if trade_direction == 1 else -y_test_raw[i]
        
        if trade_ret > take_profit:
            winning_trades += 1
            equity *= (1 + take_profit)
        elif trade_ret < stop_loss:
            equity *= (1 + stop_loss)
        else:
            if trade_ret > 0:
                winning_trades += 1
            equity *= np.exp(trade_ret)
            
        in_trade_until = i + 3
        
    equity_curve.append(equity)

win_rate = (winning_trades / trades_taken) if trades_taken > 0 else 0
baseline_return = test_closes[-1] / test_closes[0] - 1
strategy_return = equity - 1

col1, col2, col3, col4 = st.columns(4)
col1.metric("Total Trades", trades_taken)
col2.metric("Win Rate (All Profits)", f"{win_rate:.2%}")
col3.metric("Strategy Return", f"{strategy_return:.2%}")
col4.metric("Market Baseline", f"{baseline_return:.2%}")

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, 
                    vertical_spacing=0.05, row_heights=[0.7, 0.3])

fig.add_trace(plotly_go.Scatter(x=test_dates, y=test_closes, name="XAUUSD Close", line=dict(color="gray")), row=1, col=1)

# Plot BUY signals
buy_dates = [test_dates[i] for i in buy_signals_idx]
buy_prices = [test_closes[i] for i in buy_signals_idx]
marker_color_buy = "green" if "LSTM" in selected_model else "orange"

fig.add_trace(plotly_go.Scatter(x=buy_dates, y=buy_prices, mode="markers",
                                marker=dict(symbol="triangle-up", size=10, color=marker_color_buy),
                                name=f"BUY Signal"), row=1, col=1)

# Plot SELL signals
if len(sell_signals_idx) > 0:
    sell_dates = [test_dates[i] for i in sell_signals_idx]
    sell_prices = [test_closes[i] for i in sell_signals_idx]
    
    fig.add_trace(plotly_go.Scatter(x=sell_dates, y=sell_prices, mode="markers",
                                    marker=dict(symbol="triangle-down", size=10, color="red"),
                                    name=f"SELL Signal"), row=1, col=1)

fig.add_trace(plotly_go.Scatter(x=test_dates, y=equity_curve, name="Cumulative Profit", line=dict(color="blue")), row=2, col=1)

fig.update_layout(height=800, margin=dict(l=0, r=0, t=30, b=0), showlegend=True, 
                  title_text=f"Performance: {selected_model}")
st.plotly_chart(fig, use_container_width=True)

In [ ]:
!pip install -q streamlit
!pip install -q plotly

import time

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared


!pkill -f streamlit
!pkill -f cloudflared
time.sleep(2)

get_ipython().system_raw('streamlit run app.py --server.port 8501 &')

time.sleep(6)


get_ipython().system_raw('./cloudflared tunnel --url http://localhost:8501 > cloudflare_url.txt 2>&1 &')
time.sleep(5)


!grep -o 'https://.*\.trycloudflare.com' cloudflare_url.txt